# Premier League ML Predictor — Data Pipeline
MSE 446 | Phase 1: Data Collection & Exploration

## 1. Imports & Setup

In [ ]:
import requests
import pandas as pd
from io import StringIO


#Makes sure can see all columns and 50 rows before truncating
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

## 2. Data Collection

Source: [football-data.co.uk](https://www.football-data.co.uk/englandm.php)  
URL pattern: `https://www.football-data.co.uk/mmz4281/{YYYY}/E0.csv`  
Columns we keep: `Date, HomeTeam, AwayTeam, FTHG, FTAG, FTR, HS, AS, HST, AST, HC, AC`

FTHG — Full Time Home Goals
FTAG — Full Time Away Goals
FTR — Full Time Result (H/D/A)
HS — Home Shots
AS — Away Shots
HST — Home Shots on Target
AST — Away Shots on Target
HC — Home Corners
AC — Away Corners

In [42]:
# Seasons to collect: (display_name, url_code)
SEASONS = [
    ("2014-15", "1415"),
    ("2015-16", "1516"),
    ("2016-17", "1617"),
    ("2017-18", "1718"),
    ("2018-19", "1819"),
    ("2019-20", "1920"),
    ("2020-21", "2021"),
    ("2021-22", "2122"),
    ("2022-23", "2223"),
    ("2023-24", "2324"),
    ("2024-25", "2425"),
]

COLS = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "HS", "AS", "HST", "AST", "HC", "AC"]

all_seasons = []

for season_name, code in SEASONS:
    url = f"https://www.football-data.co.uk/mmz4281/{code}/E0.csv"
    print(f"Downloading {season_name}... ", end="")
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        df = pd.read_csv(StringIO(response.text), usecols=COLS)
        df["season"] = season_name
        all_seasons.append(df)
        print(f"{len(df)} rows")
    except Exception as e:
        print(f"FAILED: {e}")

raw = pd.concat(all_seasons, ignore_index=True)
print(f"\nTotal rows: {len(raw)}")


Total rows: 4181


In [24]:
raw.head(10)

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HST,AST,HC,AC,season
0,16/08/14,Arsenal,Crystal Palace,2.0,1.0,H,14.0,4.0,6.0,2.0,9.0,3.0,2014-15
1,16/08/14,Leicester,Everton,2.0,2.0,D,11.0,13.0,3.0,3.0,3.0,6.0,2014-15
2,16/08/14,Man United,Swansea,1.0,2.0,A,14.0,5.0,5.0,4.0,4.0,0.0,2014-15
3,16/08/14,QPR,Hull,0.0,1.0,A,19.0,11.0,6.0,4.0,8.0,9.0,2014-15
4,16/08/14,Stoke,Aston Villa,0.0,1.0,A,12.0,7.0,2.0,2.0,2.0,8.0,2014-15
5,16/08/14,West Brom,Sunderland,2.0,2.0,D,10.0,7.0,5.0,2.0,6.0,3.0,2014-15
6,16/08/14,West Ham,Tottenham,0.0,1.0,A,18.0,10.0,4.0,4.0,8.0,5.0,2014-15
7,17/08/14,Liverpool,Southampton,2.0,1.0,H,12.0,12.0,5.0,6.0,2.0,6.0,2014-15
8,17/08/14,Newcastle,Man City,0.0,2.0,A,12.0,13.0,0.0,5.0,3.0,3.0,2014-15
9,18/08/14,Burnley,Chelsea,1.0,3.0,A,9.0,11.0,2.0,3.0,4.0,3.0,2014-15


In [25]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4181 entries, 0 to 4180
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      4180 non-null   object 
 1   HomeTeam  4180 non-null   object 
 2   AwayTeam  4180 non-null   object 
 3   FTHG      4180 non-null   float64
 4   FTAG      4180 non-null   float64
 5   FTR       4180 non-null   object 
 6   HS        4180 non-null   float64
 7   AS        4180 non-null   float64
 8   HST       4180 non-null   float64
 9   AST       4180 non-null   float64
 10  HC        4180 non-null   float64
 11  AC        4180 non-null   float64
 12  season    4181 non-null   object 
dtypes: float64(8), object(5)
memory usage: 424.8+ KB


In [26]:
# check for missing values per column
raw.isnull().sum()

Date        1
HomeTeam    1
AwayTeam    1
FTHG        1
FTAG        1
FTR         1
HS          1
AS          1
HST         1
AST         1
HC          1
AC          1
season      0
dtype: int64

In [27]:
# All unique team names — check for inconsistencies
all_teams = pd.Series(
    pd.concat([raw["HomeTeam"], raw["AwayTeam"]]).unique()
).sort_values().reset_index(drop=True)
print(all_teams.tolist())

['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Cardiff', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Huddersfield', 'Hull', 'Ipswich', 'Leeds', 'Leicester', 'Liverpool', 'Luton', 'Man City', 'Man United', 'Middlesbrough', 'Newcastle', 'Norwich', "Nott'm Forest", 'QPR', 'Sheffield United', 'Southampton', 'Stoke', 'Sunderland', 'Swansea', 'Tottenham', 'Watford', 'West Brom', 'West Ham', 'Wolves', nan]


In [ ]:
raw[raw.isnull().any(axis=1)]
#the 2014-15 season has an extra row at the end with just nulls we can drop it. 

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HST,AST,HC,AC,season
380,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014-15


In [ ]:
df = raw.dropna(subset=["HomeTeam"]).copy()
print(len(df))  # should be 4180 total rows after dropping one


4180


In [38]:
df.head(383)

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HST,AST,HC,AC,season
0,16/08/14,Arsenal,Crystal Palace,2.0,1.0,H,14.0,4.0,6.0,2.0,9.0,3.0,2014-15
1,16/08/14,Leicester,Everton,2.0,2.0,D,11.0,13.0,3.0,3.0,3.0,6.0,2014-15
2,16/08/14,Man United,Swansea,1.0,2.0,A,14.0,5.0,5.0,4.0,4.0,0.0,2014-15
3,16/08/14,QPR,Hull,0.0,1.0,A,19.0,11.0,6.0,4.0,8.0,9.0,2014-15
4,16/08/14,Stoke,Aston Villa,0.0,1.0,A,12.0,7.0,2.0,2.0,2.0,8.0,2014-15
...,...,...,...,...,...,...,...,...,...,...,...,...,...
378,24/05/15,Newcastle,West Ham,2.0,0.0,H,17.0,4.0,4.0,1.0,2.0,3.0,2014-15
379,24/05/15,Stoke,Liverpool,6.0,1.0,H,15.0,13.0,9.0,4.0,3.0,9.0,2014-15
381,08/08/2015,Bournemouth,Aston Villa,0.0,1.0,A,11.0,7.0,2.0,3.0,6.0,3.0,2015-16
382,08/08/2015,Chelsea,Swansea,2.0,2.0,D,11.0,18.0,3.0,10.0,4.0,8.0,2015-16
